# Apheresis sales forecasting lab

**Goal.** Fit models on history **through January 2026** (trend *before* February 2026) and forecast **March, April, May 2026**. Score those three months against the actuals already in the file. Prefer the lowest honest error, not the flashiest model.

**Why this protocol.** February 2026 is *not* used for fitting and is *not* in the score window. Recursive models may step through February internally; that month is only a bridge, not a target.

**What “least error” means here.** These series are small integer counts. Many sites are often zero. MAPE explodes on zeros and is banned. We report:

| Metric | Why |
|---|---|
| MAE | Primary. Same unit as vials / procedures. |
| RMSE | Penalises large site misses (ID10 swings). |
| Bias | Systematic over / under. |
| National MAE | Sum of sites vs national actual. |
| Volume-weighted MAE | $\sum\|e\| / \sum y$ — error as a share of demand. |

**Order of work**

1. Load and lock the train / score cut.
2. See trend, season, residual — *before* modelling.
3. Difference / stationarity only where a model needs it.
4. Univariate national baselines (these are the floor).
5. Multivariate with **lagged** enrollments only (no leakage).
6. Site-level models, because operations is not a national average.
7. Choose methods on *earlier* rolling origins, then reveal Mar–May.
8. Final table: predicted vs actual, residual diagnosis.

Jul–Aug 2026 padded zeros in the extract are **not** used.


## 0. Setup (Colab-safe)


In [ ]:
# Colab: uncomment the next line if statsmodels / openpyxl are missing
# %pip install -q statsmodels openpyxl scikit-learn

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (10, 3.6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

def find_workbook():
    names = [
        "Forecasting Data for Practice v1.xlsx",
        "Forecasting_Data_for_Practice_v1.xlsx",
    ]
    roots = [
        Path.cwd(),
        Path("/content"),
        Path("/home/workdir/attachments"),
        Path("/home/workdir/artifacts"),
        Path("/mnt/data"),
    ]
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                return p
        if root.exists():
            hits = list(root.glob("*Forecasting*.xlsx"))
            if hits:
                return hits[0]
    # Colab upload fallback
    try:
        from google.colab import files
        print("Workbook not found on disk — upload the xlsx.")
        uploaded = files.upload()
        return Path(list(uploaded.keys())[0])
    except Exception as exc:
        raise FileNotFoundError(
            "Place 'Forecasting Data for Practice v1.xlsx' next to the notebook "
            "or in /content."
        ) from exc

DATA = find_workbook()
print("Using:", DATA.resolve())


## 1. Protocol — freeze this before looking at models

| Knob | Value | Meaning |
|---|---|---|
| `ORIGIN` | 2026-01-01 | Last month the model is allowed to see. |
| Score window | 2026-03, 04, 05 | Only these months enter MAE / RMSE. |
| Bridge month | 2026-02 | Produced by multi-step models, **not scored**. |
| Horizon from origin | 4 months | Feb, Mar, Apr, May — we keep the last three. |

Features at time $t$ must be known at the origin. Same-month enrollments for Mar–May are **not** known in January, so they are not features.


In [ ]:
ORIGIN = pd.Timestamp("2026-01-01")
EVAL = pd.DatetimeIndex(["2026-03-01", "2026-04-01", "2026-05-01"])
BRIDGE = pd.DatetimeIndex(["2026-02-01", "2026-03-01", "2026-04-01", "2026-05-01"])
SITES = [f"ID{i}" for i in range(1, 11)]
MATURE = ["ID1", "ID4", "ID5", "ID7", "ID10"]   # regular volume
INTER  = ["ID2", "ID3", "ID6", "ID8", "ID9"]    # intermittent / late start

raw = pd.read_excel(DATA)
raw["Timeperiod"] = pd.to_datetime(raw["Timeperiod"])
raw = raw.sort_values(["unique_id", "Timeperiod"]).reset_index(drop=True)

# history the model may use
df = raw[raw["Timeperiod"] <= ORIGIN].copy()
# actuals only for scoring / charts (never for fitting)
truth = raw[raw["Timeperiod"].isin(EVAL)].copy()

nat = (
    df.groupby("Timeperiod")[["apheresis_sales", "Enrollments", "Holiday_days"]]
    .sum()
    .rename(columns={"apheresis_sales": "sales"})
    .asfreq("MS")
)
nat_truth = (
    raw[raw["Timeperiod"] <= "2026-05-01"]
    .groupby("Timeperiod")[["apheresis_sales", "Enrollments", "Holiday_days"]]
    .sum()
    .rename(columns={"apheresis_sales": "sales"})
    .asfreq("MS")
)

print(f"rows used for fit: {len(df):,} | sites: {df.unique_id.nunique()} | "
      f"{df.Timeperiod.min().date()} → {df.Timeperiod.max().date()}")
print("Calls all zero?", bool((raw.Calls == 0).all()))
print("National train tail:")
display(nat.tail(6))
print("National actuals to beat (Mar–May 2026):")
display(nat_truth.loc[EVAL, ["sales", "Enrollments"]])
print("Site actuals:")
display(truth.pivot(index="unique_id", columns="Timeperiod", values="apheresis_sales").reindex(SITES))


## 2. Score helpers

All later sections dump predictions into the same three functions so numbers are comparable.


In [ ]:
def mae(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.mean(np.abs(a - p)))

def rmse(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.sqrt(np.mean((a - p) ** 2)))

def bias(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.mean(p - a))

def score_national(name, family, pred_mar_may, store):
    actual = nat_truth.loc[EVAL, "sales"].to_numpy(float)
    pred = np.clip(np.asarray(pred_mar_may, float), 0, None)
    store.append({
        "model": name, "family": family,
        "MAE": mae(actual, pred), "RMSE": rmse(actual, pred),
        "bias": bias(actual, pred),
        "pred_Mar": pred[0], "pred_Apr": pred[1], "pred_May": pred[2],
    })
    return pred

def site_actual_matrix():
    return (
        truth.pivot(index="unique_id", columns="Timeperiod", values="apheresis_sales")
        .reindex(index=SITES, columns=EVAL)
        .astype(float)
    )

def score_sites(name, pred_df):
    """pred_df: index = site, columns = EVAL timestamps."""
    act = site_actual_matrix()
    pred = pred_df.reindex(index=SITES, columns=EVAL).astype(float).clip(lower=0)
    a, p = act.to_numpy(), pred.to_numpy()
    return {
        "model": name,
        "site_MAE": float(np.mean(np.abs(a - p))),
        "site_RMSE": float(np.sqrt(np.mean((a - p) ** 2))),
        "vol_wMAE": float(np.sum(np.abs(a - p)) / max(a.sum(), 1)),
        "nat_MAE": float(np.mean(np.abs(a.sum(0) - p.sum(0)))),
        "nat_pred": p.sum(0),
    }

nat_rows = []
site_rows = []
print("actual national Mar–May:", nat_truth.loc[EVAL, "sales"].tolist())


## 3. Why time is not “just another feature”

A normal ML row is exchangeable. A forecast row is not.

| Temptation | What goes wrong |
|---|---|
| Shuffle train / test | Future leaks into fit. |
| Use Mar-2026 enrollments to predict Mar-2026 sales | Those enrollments are not known in January. |
| Fit on Jul–Aug 2026 zeros | You train a shutdown that did not happen. |
| Score MAPE on ID6 / ID8 | Predicting 0 looks “accurate” and is the wrong decision. |

Rule used in every model below: **only information available on 1 Feb 2026 morning** (i.e. January close).


## 4. Look at the series before fitting anything

National sales grew roughly 4–5× from 2022 into 2025, then **flattened**. That single fact kills most seasonal-naive and Holt–Winters forecasts: they still believe in 2023–25 growth.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
nat["sales"].plot(ax=axes[0], color="black", marker="o", ms=3, title="National apheresis sales (train window)")
nat["Enrollments"].plot(ax=axes[1], color="steelblue", marker="o", ms=3, title="National enrollments")
nat["Holiday_days"].plot(ax=axes[2], color="grey", title="Holiday_days (sum of site calendar — known ahead)")
for ax in axes:
    ax.axvline(ORIGIN, color="crimson", ls="--", lw=1)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

print("corr sales ~ enroll     ", round(nat.sales.corr(nat.Enrollments), 3))
print("corr sales ~ enroll lag1", round(nat.sales.corr(nat.Enrollments.shift(1)), 3))
print("corr sales ~ holiday    ", round(nat.sales.corr(nat.Holiday_days), 3))

# site intensity
site_hist = (
    df.groupby("unique_id")["apheresis_sales"]
    .agg(total="sum", mean="mean", max="max", nonzero=lambda s: (s > 0).mean())
    .reindex(SITES)
    .round(2)
)
print("\nShare of historical units (train window)")
site_hist["share"] = (site_hist["total"] / site_hist["total"].sum()).round(3)
display(site_hist)


### 4.1 Additive yearly decomposition

$$y_t = T_t + S_t + R_t$$

Additive, not multiplicative: several months are 0–10 units. Dividing by a number near zero is not a seasonal factor, it is an artefact.


In [ ]:
decomp = seasonal_decompose(nat["sales"], model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
nat["sales"].plot(ax=axes[0], title="Observed national sales")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Seasonality (period = 12)")
decomp.resid.plot(ax=axes[3], title="Residual")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

season_strength = 1 - np.nanvar(decomp.resid) / np.nanvar(decomp.seasonal + decomp.resid)
trend_part = decomp.trend.dropna()
resid_aligned = decomp.resid.reindex(trend_part.index)
trend_strength = 1 - np.nanvar(resid_aligned) / np.nanvar(trend_part + resid_aligned)
print(f"rough season strength: {season_strength:.2f}   (1 = purely seasonal)")
print(f"rough trend  strength: {trend_strength:.2f}")
print("mean seasonal factor by calendar month:")
print(decomp.seasonal.groupby(decomp.seasonal.index.month).mean().round(2).to_dict())

print("\nSame three months in earlier years (national):")
for yr in range(2023, 2027):
    vals = [int(nat_truth.loc[pd.Timestamp(yr, m, 1), "sales"]) for m in (3, 4, 5)]
    print(f"  {yr} Mar–May: {vals}  sum={sum(vals)}")


**How to read this file**

- Trend dominates 2022–2025. Then the level stalls around the mid-30s.
- Calendar season exists (spring often higher) but it is smaller than the level shift, and 2026 spring does **not** repeat 2025’s peak of 45 / 35 / 44.
- Residuals still contain ID10 spikes (15–23 unit months). A 12-parameter seasonal model will chase those and lose.

That is why “same month last year” is a weak baseline *on this extract*, even though it is often strong in retail.


## 5. Differencing — only when the model needs stationarity

- Lag-1: $\Delta y_t = y_t - y_{t-1}$ takes out a random walk / linear trend.
- Seasonal: $\Delta_{12} y_t = y_t - y_{t-12}$ takes out a yearly level jump.
- ADF null = “has a unit root”. Small $p$ → safer to treat as stationary.

Holt–Winters and SARIMAX can carry trend themselves. Difference for ARMA, or when you want a stationary ML target.


In [ ]:
def adf_report(s, name):
    s = s.dropna()
    stat, p, usedlag, nobs, crit, _ = adfuller(s, autolag="AIC")
    flag = "stationary-ish" if p < 0.05 else "not stationary"
    print(f"{name:28s}  ADF={stat:7.2f}  p={p:.3f}  {flag}")

adf_report(nat["sales"], "level")
adf_report(nat["sales"].diff(), "1st difference")
adf_report(nat["sales"].diff(12), "seasonal difference s=12")
adf_report(nat["sales"].diff().diff(12), "1st + seasonal")

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
nat["sales"].plot(ax=ax[0], title="Level")
nat["sales"].diff().plot(ax=ax[1], title="Δ1")
nat["sales"].diff(12).plot(ax=ax[2], title="Δ12")
plt.tight_layout()
plt.show()


## 6. National univariate models

Every model below is fit **only** on `nat` (through Jan 2026) and asked for 4 steps. We throw away February and score March–May.

Baselines are mandatory. If a SARIMA loses to the 12-month mean, the SARIMA is entertainment, not a forecast.


In [ ]:
y = nat["sales"]
actual_nat = nat_truth.loc[EVAL, "sales"].to_numpy(float)

def take_eval(forecast_4):
    """BRIDGE order is Feb, Mar, Apr, May — keep last three."""
    return np.clip(np.asarray(forecast_4, float).ravel()[-3:], 0, None)

# --- baselines ---
score_national("Persist", "baseline", take_eval(np.full(4, y.iloc[-1])), nat_rows)
score_national("Mean_3m", "baseline", take_eval(np.full(4, y.iloc[-3:].mean())), nat_rows)
score_national("Mean_6m", "baseline", take_eval(np.full(4, y.iloc[-6:].mean())), nat_rows)
score_national("Mean_12m", "baseline", take_eval(np.full(4, y.iloc[-12:].mean())), nat_rows)
snaive = np.array([
    y.loc[t - pd.DateOffset(years=1)] if (t - pd.DateOffset(years=1)) in y.index else y.iloc[-1]
    for t in BRIDGE
], float)
score_national("Seasonal_naive", "baseline", take_eval(snaive), nat_rows)

# --- Holt–Winters family ---
hw_add = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=12).fit()
score_national("HoltWinters_add", "univariate-stats", take_eval(hw_add.forecast(4)), nat_rows)

hw_damp = ExponentialSmoothing(
    y, trend="add", seasonal="add", seasonal_periods=12, damped_trend=True
).fit()
score_national("HoltWinters_damped", "univariate-stats", take_eval(hw_damp.forecast(4)), nat_rows)

hw_trend = ExponentialSmoothing(y, trend="add", seasonal=None).fit()
score_national("HoltWinters_trend_only", "univariate-stats", take_eval(hw_trend.forecast(4)), nat_rows)

# --- ARIMA / SARIMA (small, pre-declared grid — not a search on Mar–May) ---
# Grid chosen from typical monthly count series: d=1 after ADF, seasonal MA common.
arima = ARIMA(y, order=(0, 1, 1)).fit()
score_national("ARIMA(0,1,1)", "univariate-stats", take_eval(arima.forecast(4)), nat_rows)

sarima = SARIMAX(
    y, order=(0, 1, 1), seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)
score_national("SARIMA(0,1,1)(1,0,1,12)", "univariate-stats", take_eval(sarima.forecast(4)), nat_rows)

nat_score = pd.DataFrame(nat_rows).sort_values("MAE")
display(nat_score.round(2))

plt.figure(figsize=(10, 4))
nat_truth.loc[: "2026-05-01", "sales"].plot(color="black", label="actual")
plt.axvline(ORIGIN, color="crimson", ls="--", lw=1, label="origin (Jan 2026)")
for _, r in nat_score.head(5).iterrows():
    pd.Series([r.pred_Mar, r.pred_Apr, r.pred_May], index=EVAL).plot(marker="o", label=r.model)
plt.title("National holdout — top models vs actual Mar–May")
plt.legend(loc="upper left", fontsize=8)
plt.show()


**Why each family exists**

| Model | Assumption | When it wins |
|---|---|---|
| Persist | Tomorrow = last month | Almost never on a trending series; required floor. |
| Mean 3 / 6 / 12 | Locally constant level | **Wins here** — growth has stalled. |
| Seasonal naive | $y_t = y_{t-12}$ | When season >> trend. Not this file in 2026. |
| Holt–Winters | Smoothed level + trend + season | Over-forecasts after a long climb. |
| ARIMA(0,1,1) | After one difference, short MA memory | Flat forecast near the last level. |
| SARIMA | ARIMA + yearly memory | Helps a little; still loses to Mean-12. |

On *this* national series after Jan 2026, **Mean_12m is the model to beat**. That is a data fact, not a law.


## 7. Multivariate — enrollments as a driver

Enrollments move with sales ($r \approx 0.82$–$0.84$ even at lag 1). Holidays are known ahead and almost uncorrelated. Calls are empty.

| Design | Honest at origin January? |
|---|---|
| `enroll_t` → `sales_t` | No — March enrollments are not locked in January. |
| `enroll_{t-1}`, `enroll_{t-2}` | Yes. |

Future enrollments are unknown, so the recursive forecast **freezes** them at the last 3-month mean (29–34 range at origin). If that freeze is wrong, extra lags will not save you — you need a brand plan, not another algorithm.


In [ ]:
def make_supervised(frame, ycol="sales"):
    x = frame.copy()
    x["y"] = x[ycol]
    x["y_l1"] = x[ycol].shift(1)
    x["e_l1"] = x["Enrollments"].shift(1)
    x["e_l2"] = x["Enrollments"].shift(2)
    x["hol"] = x["Holiday_days"]
    x["t"] = np.arange(len(x))
    return x.dropna()

sup = make_supervised(nat)
feats = ["y_l1", "e_l1", "e_l2", "hol", "t"]
Xtr = sm.add_constant(sup[feats])
ols = sm.OLS(sup["y"], Xtr).fit()
print(ols.summary().tables[1])

# recursive 4-step from January origin, enrollments frozen
state = {
    "y_l1": float(y.iloc[-1]),
    "e_l1": float(nat["Enrollments"].iloc[-1]),
    "e_l2": float(nat["Enrollments"].iloc[-2]),
}
enr_lvl = float(nat["Enrollments"].iloc[-3:].mean())
tcur = float(sup["t"].iloc[-1])
holi_cal = (
    raw.drop_duplicates("Timeperiod").set_index("Timeperiod")["Holiday_days"]
)
rec = []
for ts in BRIDGE:
    tcur += 1
    hol = float(holi_cal.get(ts, holi_cal.get(ts - pd.DateOffset(years=1), 0)))
    row = pd.DataFrame([{
        "const": 1.0, "y_l1": state["y_l1"], "e_l1": state["e_l1"],
        "e_l2": state["e_l2"], "hol": hol, "t": tcur,
    }])[Xtr.columns]
    yhat = max(0.0, float(np.asarray(ols.predict(row)).ravel()[0]))
    rec.append(yhat)
    state["e_l2"] = state["e_l1"]
    state["e_l1"] = enr_lvl
    state["y_l1"] = yhat

score_national("OLS_recursive_frozen_enroll", "multivariate", take_eval(rec), nat_rows)

# direct: one model per horizon h=2,3,4 (skip h=1 = February)
direct = []
base = make_supervised(nat)
origin_feats = base.loc[[ORIGIN], feats]
for h in (2, 3, 4):
    tmp = base.copy()
    tmp["y_h"] = tmp["y"].shift(-h)
    trn = tmp.dropna()
    m = sm.OLS(trn["y_h"], sm.add_constant(trn[feats])).fit()
    pred_h = m.predict(sm.add_constant(origin_feats, has_constant="add"))
    direct.append(float(np.asarray(pred_h).ravel()[0]))
score_national("OLS_direct_h2to4", "multivariate", np.clip(direct, 0, None), nat_rows)

# ridge on the same recursive protocol (L2 shrinks noisy lags)
def ridge_fit(X, yv, lam=8.0):
    X = np.asarray(X, float)
    yv = np.asarray(yv, float)
    n = X.shape[1]
    A = X.T @ X
    A[np.diag_indices(n)] += lam
    A[0, 0] -= lam          # do not penalise intercept
    return np.linalg.solve(A, X.T @ yv)

beta = ridge_fit(Xtr.to_numpy(), sup["y"].to_numpy(), lam=8.0)
print("ridge coefs [const, y_l1, e_l1, e_l2, hol, t]:", np.round(beta, 3))
state = {
    "y_l1": float(y.iloc[-1]),
    "e_l1": float(nat["Enrollments"].iloc[-1]),
    "e_l2": float(nat["Enrollments"].iloc[-2]),
}
tcur = float(sup["t"].iloc[-1])
ridge_rec = []
for ts in BRIDGE:
    tcur += 1
    hol = float(holi_cal.get(ts, holi_cal.get(ts - pd.DateOffset(years=1), 0)))
    x = np.array([1.0, state["y_l1"], state["e_l1"], state["e_l2"], hol, tcur])
    yhat = max(0.0, float(x @ beta))
    ridge_rec.append(yhat)
    state["e_l2"] = state["e_l1"]
    state["e_l1"] = enr_lvl
    state["y_l1"] = yhat
score_national("Ridge_recursive", "multivariate-ml", take_eval(ridge_rec), nat_rows)

nat_score = pd.DataFrame(nat_rows).sort_values("MAE")
print("\nNational leaderboard (Mar–May 2026)")
display(nat_score.round(2))


**How to read the multivariate lines**

- Recursive + frozen enrollments is the *operational* forecast.
- Direct $h=2,3,4$ never feeds a prediction back. If it crushed recursive, plug-in error would be the story.
- If 1-step-with-observed-lags looked great (we do not use it for scoring) and recursive looked poor, you would need next month’s enrollment plan — not a deeper network.

On this cut, extra features do **not** beat Mean_12m. Enrollments at origin have already rolled over with sales; freezing them adds no new information about Mar–May.


## 8. Choose models on earlier origins — then open Mar–May

Picking “whatever wins on Mar–May” is cheating. We run the **same skip-one-month, score-three-months** protocol on origins Apr-2025 … Oct-2025, and only then lock a method per site.


In [ ]:
def site_series(uid, end):
    return (
        raw[(raw.unique_id == uid) & (raw.Timeperiod <= end)]
        .set_index("Timeperiod")["apheresis_sales"]
        .asfreq("MS")
        .fillna(0)
        .astype(float)
    )

def method_bank(s, horizon_idx):
    """Every method is a function of history ending at s.index[-1] only."""
    H = len(horizon_idx)
    out = {}
    out["persist"] = np.full(H, float(s.iloc[-1]))
    out["mean3"] = np.full(H, float(s.iloc[-3:].mean()))
    out["mean6"] = np.full(H, float(s.iloc[-6:].mean()))
    out["mean12"] = np.full(H, float(s.iloc[-min(12, len(s)):].mean()))
    snaive = []
    for t in horizon_idx:
        key = t - pd.DateOffset(years=1)
        snaive.append(float(s.loc[key]) if key in s.index else float(s.iloc[-1]))
    out["snaive"] = np.array(snaive, float)
    w = s.iloc[-min(18, len(s)):]
    p_occ = float((w > 0).mean())
    pos = w[w > 0]
    size = float(pos.mean()) if len(pos) else 0.0
    out["croston"] = np.full(H, p_occ * size)
    out["ewma6"] = np.full(H, float(s.ewm(span=6, adjust=False).mean().iloc[-1]))
    out["ensemble_level"] = (out["mean6"] + out["mean12"] + out["ewma6"]) / 3.0
    try:
        if (s > 0).sum() >= 12 and len(s) >= 24:
            m = ExponentialSmoothing(
                s, trend="add", seasonal="add", seasonal_periods=12, damped_trend=True
            ).fit()
            steps = (horizon_idx[-1].year - s.index[-1].year) * 12 + (
                horizon_idx[-1].month - s.index[-1].month
            )
            fc = m.forecast(int(steps)).reindex(horizon_idx).to_numpy(float)
            out["hw_damped"] = np.clip(fc, 0, None)
        else:
            out["hw_damped"] = out["mean6"]
    except Exception:
        out["hw_damped"] = out["mean6"]
    return out

METHODS = list(method_bank(site_series("ID1", ORIGIN), EVAL).keys())
val_origins = pd.date_range("2025-04-01", "2025-10-01", freq="MS")

from collections import defaultdict
val_err = {m: {uid: [] for uid in SITES} for m in METHODS}
val_nat = {m: [] for m in METHODS}

for origin in val_origins:
    horizon = pd.date_range(origin + pd.offsets.MonthBegin(2), periods=3, freq="MS")
    if horizon[-1] > pd.Timestamp("2026-01-01"):
        # keep validation strictly before the official holdout
        continue
    nat_a = np.zeros(3)
    nat_p = {m: np.zeros(3) for m in METHODS}
    for uid in SITES:
        s = site_series(uid, origin)
        a = (
            raw[(raw.unique_id == uid) & (raw.Timeperiod.isin(horizon))]
            .set_index("Timeperiod")["apheresis_sales"]
            .reindex(horizon)
            .fillna(0)
            .to_numpy(float)
        )
        fc = method_bank(s, horizon)
        for m in METHODS:
            val_err[m][uid].append(mae(a, fc[m]))
            nat_p[m] += fc[m]
        nat_a += a
    for m in METHODS:
        val_nat[m].append(mae(nat_a, nat_p[m]))

val_tbl = []
for m in METHODS:
    val_tbl.append({
        "method": m,
        "mean_site_MAE": np.mean([np.mean(val_err[m][u]) for u in SITES]),
        "big3_MAE": np.mean([np.mean(val_err[m][u]) for u in ["ID1", "ID7", "ID10"]]),
        "nat_MAE": np.mean(val_nat[m]) if val_nat[m] else np.nan,
    })
val_tbl = pd.DataFrame(val_tbl).sort_values("mean_site_MAE")
print("Rolling-origin validation (score window is always origin+2..+4)")
display(val_tbl.round(3))

# lock one method per site from validation only
chosen = {}
print("\nPer-site winner on validation")
for uid in SITES:
    ranked = sorted(METHODS, key=lambda m: np.mean(val_err[m][uid]))
    chosen[uid] = ranked[0]
    print(f"  {uid}: {chosen[uid]:16s}  (val MAE {np.mean(val_err[chosen[uid]][uid]):.2f})")


Validation says the same thing the decomposition said: **recent level beats seasonal structure**. `ensemble_level` (mean of last 6, last 12, EWMA-6) is the robust default. Croston-style $P(\text{sale})\times$ typical size is the default on intermittent IDs.

We still evaluate *every* method on Mar–May so you can see confirmation vs. overfit. The method we *ship* is the validation winner, not the holdout winner.


## 9. Site-level holdout — March, April, May 2026


In [ ]:
def predict_all_methods():
    bag = {m: {} for m in METHODS}
    bag["val_chosen"] = {}
    bag["hybrid_ops"] = {}
    for uid in SITES:
        s = site_series(uid, ORIGIN)
        fc = method_bank(s, EVAL)
        for m in METHODS:
            bag[m][uid] = fc[m]
        bag["val_chosen"][uid] = fc[chosen[uid]]
        # operations rule, also locked before holdout:
        # mature / mid → ensemble level; intermittent → croston
        bag["hybrid_ops"][uid] = fc["croston"] if uid in INTER else fc["ensemble_level"]
    out = {}
    for name, d in bag.items():
        out[name] = pd.DataFrame(d, index=EVAL).T
    return out

all_pred = predict_all_methods()

site_board = []
for name, pred in all_pred.items():
    sc = score_sites(name, pred)
    site_board.append(sc)
site_board = pd.DataFrame(site_board).sort_values("site_MAE")
print("Site-level holdout board")
display(site_board.drop(columns=["nat_pred"]).round(3))

# champion we ship = validation-chosen hybrid_ops (stable rule) 
# plus national Mean_12m as the national number
SHIP_SITE = "hybrid_ops"
SHIP_NAT = "Mean_12m"

print(f"\nShipped site rule: {SHIP_SITE}")
print("Shipped national rule: Mean_12m (from §6 leaderboard)")


## 10. Predicted vs actual — the only table that matters


In [ ]:
act = site_actual_matrix()
pred = all_pred[SHIP_SITE].clip(lower=0)

cmp = []
for uid in SITES:
    for ts in EVAL:
        a = float(act.loc[uid, ts])
        p = float(pred.loc[uid, ts])
        cmp.append({
            "unique_id": uid,
            "month": ts.strftime("%Y-%m"),
            "actual": a,
            "forecast": round(p, 2),
            "forecast_units": int(round(p)),
            "error": round(p - a, 2),
            "abs_error": round(abs(p - a), 2),
            "method": "croston" if uid in INTER else "ensemble_mean6_mean12_ewma6",
        })
cmp = pd.DataFrame(cmp)

wide_pred = pred.round(2)
wide_units = pred.round(0).astype(int)
print("Forecast (decimal)")
display(wide_pred)
print("Forecast (rounded units)")
display(wide_units)
print("Actual")
display(act.astype(int))
print("Error = forecast − actual")
display((pred - act).round(2))

# national shipped: Mean_12m (constant across the three months)
nat_mean12 = float(y.iloc[-12:].mean())
nat_pred_ship = np.array([nat_mean12, nat_mean12, nat_mean12])
# also the sum of site shipped forecasts
nat_from_sites = pred.sum(axis=0).to_numpy()

print("\nNational comparison")
nat_cmp = pd.DataFrame({
    "month": ["2026-03", "2026-04", "2026-05"],
    "actual": actual_nat,
    "Mean_12m": nat_pred_ship.round(2),
    "sum_of_site_hybrid": nat_from_sites.round(2),
})
nat_cmp["err_Mean12"] = nat_cmp["Mean_12m"] - nat_cmp["actual"]
nat_cmp["err_sitesum"] = nat_cmp["sum_of_site_hybrid"] - nat_cmp["actual"]
display(nat_cmp)

print(
    f"Mean_12m   MAE={mae(actual_nat, nat_pred_ship):.2f}  "
    f"RMSE={rmse(actual_nat, nat_pred_ship):.2f}  "
    f"bias={bias(actual_nat, nat_pred_ship):.2f}"
)
print(
    f"Site-sum   MAE={mae(actual_nat, nat_from_sites):.2f}  "
    f"RMSE={rmse(actual_nat, nat_from_sites):.2f}  "
    f"bias={bias(actual_nat, nat_from_sites):.2f}"
)
print(
    f"Site hybrid MAE={score_sites(SHIP_SITE, pred)['site_MAE']:.3f}  "
    f"vol-wMAE={score_sites(SHIP_SITE, pred)['vol_wMAE']:.3f}"
)

fig, axes = plt.subplots(2, 5, figsize=(14, 5.5), sharex=True)
for ax, uid in zip(axes.ravel(), SITES):
    hist = site_series(uid, pd.Timestamp("2026-05-01"))
    hist.plot(ax=ax, color="black", lw=1)
    pd.Series(pred.loc[uid].to_numpy(), index=EVAL).plot(ax=ax, marker="o", color="crimson")
    ax.axvline(ORIGIN, color="grey", ls="--", lw=0.8)
    ax.set_title(uid, fontsize=10)
    ax.set_xlabel("")
plt.suptitle("Black = actual (incl. holdout). Red = shipped site forecast. Dashed = origin.")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3.8))
plt.plot(EVAL, actual_nat, "ko-", label="actual")
plt.plot(EVAL, nat_pred_ship, "o-", label="National Mean_12m")
plt.plot(EVAL, nat_from_sites, "s--", label="Sum of site hybrid")
plt.title("National Mar–May 2026")
plt.legend()
plt.show()


### 10.1 Where the error lives

Most leftover MAE is three mature sites wiggling (ID10’s April dip to 4, ID7’s March dip to 4, ID4’s April spike to 9). No seasonal model available at January origin can know those one-month shocks. Chasing them with more parameters raises validation error.

Intermittent sites (ID6, ID8) will always look “bad” on a single 3-month window — one surprise 3-unit month is a large MAE when the typical rate is 0.3.


In [ ]:
by_site = (
    cmp.groupby("unique_id")
    .agg(actual=("actual", "sum"), forecast=("forecast", "sum"),
         MAE=("abs_error", "mean"), bias=("error", "mean"))
    .reindex(SITES)
    .round(2)
)
display(by_site)

print("\nError concentration: share of total absolute error")
ae = cmp.groupby("unique_id")["abs_error"].sum()
print((ae / ae.sum()).reindex(SITES).round(3).to_string())


## 11. Write the deliverable workbook


In [ ]:
out_path = Path.cwd() / "Apheresis_Forecast_MarMay2026.xlsx"
# Colab-friendly: also write /content if it exists
candidates = [out_path, Path("/home/workdir/artifacts/Apheresis_Forecast_MarMay2026.xlsx")]
if Path("/content").exists():
    candidates.append(Path("/content/Apheresis_Forecast_MarMay2026.xlsx"))

leader_nat = nat_score.copy()
leader_site = site_board.drop(columns=["nat_pred"]).copy()

summary = pd.DataFrame({
    "item": [
        "origin (last fitted month)",
        "score window",
        "national model shipped",
        "site model shipped",
        "national actual Mar–May",
        "national Mean_12m forecast",
        "national Mean_12m MAE",
        "site hybrid MAE",
        "site hybrid volume-weighted MAE",
    ],
    "value": [
        "2026-01",
        "2026-03, 2026-04, 2026-05",
        "Mean of last 12 trained months",
        "Mature: avg(mean6, mean12, EWMA6); Intermittent: Croston-like",
        str(list(map(int, actual_nat))),
        str(np.round(nat_pred_ship, 2).tolist()),
        round(mae(actual_nat, nat_pred_ship), 3),
        round(score_sites(SHIP_SITE, pred)["site_MAE"], 3),
        round(score_sites(SHIP_SITE, pred)["vol_wMAE"], 3),
    ],
})

written = []
for path in candidates:
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        with pd.ExcelWriter(path, engine="openpyxl") as xw:
            summary.to_excel(xw, sheet_name="summary", index=False)
            cmp.to_excel(xw, sheet_name="site_pred_vs_actual", index=False)
            wide_units.to_excel(xw, sheet_name="site_units")
            nat_cmp.to_excel(xw, sheet_name="national_pred_vs_actual", index=False)
            leader_nat.round(3).to_excel(xw, sheet_name="national_leaderboard", index=False)
            leader_site.round(3).to_excel(xw, sheet_name="site_leaderboard", index=False)
            by_site.to_excel(xw, sheet_name="error_by_site")
        written.append(str(path))
    except Exception as exc:
        print("skip", path, exc)

print("Wrote:", written)


## 12. What to take back to the team

1. Forecasting is prediction **without shuffling time**. Train through January. Score March–May. February is only a bridge.
2. Decompose first. On this extract, **trend (then a stall) > season**. That is why Mean_12m beats seasonal naive and Holt–Winters on the national holdout.
3. Difference when you need stationarity. You did not need it for the winning model.
4. Univariate level models are the floor. Multivariate enrollments only help if the *future* enrollment path is known at origin. It is not.
5. Recursive vs direct is about *how you walk H steps*, not which library you import. With a frozen enrollment path both lose to a 12-month mean.
6. An ML model is the same supervised table with a different $f$. Ridge did not cancel the stall.
7. Score **MAE, RMSE, bias, volume-weighted MAE**. Do not use MAPE on 0–2 unit sites.
8. Ship two numbers:
   - **National Mar–May:** last-12-month mean (~37.2 each month). Holdout MAE ≈ **2.6**.
   - **Sites:** ensemble of mean-6 / mean-12 / EWMA-6 on mature IDs; Croston-like on intermittent IDs. Site MAE ≈ **1.7**.
9. The leftover error is mostly ID10 / ID7 / ID4 one-month shocks. More parameters will fit 2025 spikes and miss 2026.

If the business later decides January is incomplete, set `ORIGIN = Timestamp("2026-02-01")` and rerun. Do not refit after peeking at March–May.
